### 检验manifest中dev的数据

In [10]:
import pandas as pd

manifest = pd.read_csv(
    r"D:\Projects\speech-ai-learning\metadata\manifest.csv"
)

dev = manifest[manifest["split"] == "dev"].copy()

print("Dev 样本数量：", len(dev))
print("字段：", dev.columns.tolist())
print(dev.head())

Dev 样本数量： 409
字段： ['audio_path', 'speaker_id', 'text', 'source_id', 'parent_uid', 'start_time', 'end_time', 'split']
                           audio_path  \
2281  common_voice_zh-CN_18600769.mp3   
2282  common_voice_zh-CN_18600881.mp3   
2283  common_voice_zh-CN_18600767.mp3   
2284  common_voice_zh-CN_18600768.mp3   
2285  common_voice_zh-CN_18641893.mp3   

                                             speaker_id  \
2281  5d66e9ec552b7b74ae9c1137559ea3cdfe341af1e3f898...   
2282  5d66e9ec552b7b74ae9c1137559ea3cdfe341af1e3f898...   
2283  5d66e9ec552b7b74ae9c1137559ea3cdfe341af1e3f898...   
2284  5d66e9ec552b7b74ae9c1137559ea3cdfe341af1e3f898...   
2285  5d66e9ec552b7b74ae9c1137559ea3cdfe341af1e3f898...   

                          text                     source_id  \
2281  尤因镇区是位于美国阿肯色州布恩县的一个行政镇区。  common_voice_beijing26_zh-CN   
2282  莫尔特里是一个位于美国乔治亚州科尔奎特县的城市。  common_voice_beijing26_zh-CN   
2283                 无法联系或拒绝电访  common_voice_beijing26_zh-CN   
2284               一级造园施工

### 创建dev-candidates 409条候选队列

In [2]:
#创建候选队列
from pathlib import Path
import pandas as pd
import hashlib

# 路径
root = Path(r"D:\Projects\speech-ai-learning")

manifest_path = root / "metadata" / "manifest.csv"
clips_dir = (
    root / "data" / "raw" /
    "common_voice_beijing26" /
    "zh-CN-beijing" / "clips"
)

output_path = (
    root / "metadata" / "dev_candidates_W7D5.csv"
)

#防止覆盖已有文件
assert not output_path.exists(), (
    f"候选文件已经存在：{output_path}"
)

#读取Dev
manifest = pd.read_csv(manifest_path)
dev = manifest[
    manifest["split"] == "dev"
].copy()

assert len(dev) == 409

#检查必要字段
required_columns = [
    "audio_path",
    "speaker_id",
    "text",
    "source_id",
    "parent_uid",
    "start_time",
    "end_time",
    "split"
]

assert all(
    col in dev.columns
    for col in required_columns
)

assert dev["parent_uid"].notna().all()
assert dev["audio_path"].notna().all()

# 检查 UID 是否重复
assert dev["parent_uid"].is_unique

# =====================
# 3. 固定排序后随机打乱
# =====================

# 先按照 parent_uid 排序，
# 确保随机打乱前的输入顺序固定

dev = dev.sort_values(
    "parent_uid"
).reset_index(drop=True)

# 全部409条随机打乱一次
# 固定随机种子，保证可复现

dev_candidates = dev.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

#分配固定selection_rank
dev_candidates["selection_rank"] = range(
    1,
    len(dev_candidates) + 1
)


# =====================
# 4. 构建候选记录
# =====================

records = []

for _, row in dev_candidates.iterrows():

    #音频文件名
    audio_name = Path(
        row["audio_path"]
    ).name

    # 完整音频路径
    audio_path = clips_dir / audio_name

    # 检查音频是否存在
    assert audio_path.exists(), (
        f"音频不存在：{audio_path}"
    )

    # 根据原始 audio_path 生成稳定 UID
    uid_hash = hashlib.sha256(
        str(row["audio_path"]).encode("utf-8")
    ).hexdigest()[:16]

    candidate_uid = f"cv26_{uid_hash}"

    # 音频时长
    duration_sec = (
        row["end_time"] - row["start_time"]
    )

    records.append({

        "selection_rank": (
            len(records) + 1
        ),

        "candidate_uid": candidate_uid,

        "audio_path": str(audio_path),

        "duration_sec": duration_sec,

        "speaker_id": row["speaker_id"],

        "parent_uid": row["parent_uid"],

        "source_text": row["text"],

        "raw_text": "",

        "score_text": "",

        "status": "pending",

        "review_minutes": "",

        "split": "dev"
    })

# =====================
# 5. 转换为 DataFrame
# =====================

dev_candidates = pd.DataFrame(records)

# =====================
# 6. 完整性检查
# =====================

assert len(dev_candidates) == 409

assert dev_candidates["candidate_uid"].is_unique

assert dev_candidates["parent_uid"].is_unique

assert (
    dev_candidates["status"] == "pending"
).all()

assert (
    dev_candidates["split"] == "dev"
).all()

assert dev_candidates["audio_path"].apply(
    lambda x: Path(x).exists()
).all()


# =====================
# 7. 保存候选队列
# =====================

dev_candidates.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# 8. 输出检查结果
# =====================

print("W7D5 Dev 候选队列创建成功")

print("候选数量:", len(dev_candidates))

print("\n状态统计:")
print(
    dev_candidates["status"].value_counts()
)

print("\n字段检查:")
for col in [
    "raw_text",
    "score_text",
    "review_minutes"
]:
    print(
        f"{col} 为空:",
        dev_candidates[col].isna().sum()
        + (dev_candidates[col] == "").sum()
    )

print("\n前10条候选:")
print(
    dev_candidates[
        [
            "selection_rank",
            "candidate_uid",
            "audio_path",
            "duration_sec",
            "status"
        ]
    ].head(10).to_string(index=False)
)

print("\n已保存:", output_path)

print("\nDev 候选队列完整性检查通过")

W7D5 Dev 候选队列创建成功
候选数量: 409

状态统计:
status
pending    409
Name: count, dtype: int64

字段检查:
raw_text 为空: 409
score_text 为空: 409
review_minutes 为空: 409

前10条候选:
 selection_rank         candidate_uid                                                                                                         audio_path  duration_sec  status
              1 cv26_3f66c228bc2a4ae8 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22057623.mp3      5.957625 pending
              2 cv26_1753be3b6083f28f D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22004281.mp3      2.789625 pending
              3 cv26_20558b6a5c8dbac9 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22004016.mp3      5.621625 pending
              4 cv26_40123f9681ffefa4 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_18

### 创建音频展示函数和标注更新函数

In [1]:
#读取音频并展示
from pathlib import Path
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

root = Path.cwd().resolve()
if root.name.lower() == "notebooks":
    root = root.parent

candidate_path = root / "metadata" / "dev_candidates_W7D5.csv"

audio_root = (
    root / "data" / "raw"
    / "common_voice_beijing26"
    / "zh-CN-beijing"
)

def resolve_audio_path(audio_path):
    p = Path(str(audio_path))

    candidates = (
        [p] if p.is_absolute() else [
            root / p,
            audio_root / p,
            audio_root / "clips" / p
        ]
    )

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"找不到音频:{audio_path}"
    )

def show_candidate(rank):
    df = pd.read_csv(
        candidate_path,
        keep_default_na=False
    )

    selected = df.loc[
        df["selection_rank"] == rank
    ]

    if len(selected) != 1:
        raise ValueError("没有找到唯一候选样本")

    row = selected.iloc[0]

    #if row["review_batch"] != "W5D4" and row["review_batch"] != "W5D5":
        #raise ValueError("该样本不属于W5D4/W5D5")

    path = resolve_audio_path(row["audio_path"])

    #读取音频
    audio, sr = sf.read(
        path,
        dtype="float32",
        always_2d=True
    )

    #如果有多个声道，简单取平均
    waveform = audio.mean(axis=1)

    print("序号:", rank)
    print("UID:", row["candidate_uid"])
    print("音频时长:", len(waveform) / sr)
    print("当前状态:", row["status"])
    print("原始转写:", row["source_text"])

    display(Audio(waveform, rate=sr))

In [2]:
#标注更新函数
def update_candidate(
        rank,
        raw_text,
        status,
        score_text="",
        uncertain_span="",
        notes="",
        review_minutes=""
):
    if status not in ["verified","needs_review"]:
        raise ValueError("无效标注状态")

    if status == "verified":
        if not str(raw_text).strip():
            raise ValueError("verified必须有确认后的文本")

        if str(uncertain_span).strip():
            raise ValueError(
                "仍存在不确定的片段，不能标记verified"
            )

    if status == "needs_review":
        if not str(uncertain_span).strip() and not str(notes).strip():
            raise ValueError("请记录需要复核的原因")

    df = pd.read_csv(
        candidate_path,
        dtype=str,
        keep_default_na=False
    )

    mask = df["selection_rank"] == str(rank)

    if mask.sum() != 1:
        raise ValueError("没有找到唯一候选样本")


    df.loc[mask, "raw_text"] = raw_text
    df.loc[mask, "score_text"] = score_text
    df.loc[mask, "status"] = status
    df.loc[mask, "uncertain_span"] = uncertain_span
    df.loc[mask, "notes"] = notes
    df.loc[mask, "review_minutes"] = str(review_minutes)

    #临时文件写入成功后，再替换原CSV
    temp_path = candidate_path.with_suffix(".csv.tmp")

    df.to_csv(
        temp_path,
        index=False,
        encoding="utf-8-sig"
    )

    temp_path.replace(candidate_path)

    print(f"第{rank}条已更新为：{status}")

### 2.更新1-10条候选队列

In [5]:
#检验队列
import pandas as pd

path = (
    r"D:\Projects\speech-ai-learning"
    r"\metadata\dev_candidates_W7D5.csv"
)

df = pd.read_csv(path)

batch = (
    df[df["status"] == "pending"]
    .head(10)
)

print(batch[
    [
        "selection_rank",
        "candidate_uid",
        "audio_path",
        "duration_sec",
        "source_text",
        "status"
    ]
].to_string(index=False))

assert len(batch) == 10

 selection_rank         candidate_uid                                                                                                         audio_path  duration_sec                           source_text  status
              1 cv26_3f66c228bc2a4ae8 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22057623.mp3      5.957625                       这两家公司都在网上出售树莓派。 pending
              2 cv26_1753be3b6083f28f D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22004281.mp3      2.789625                                武藏国出身。 pending
              3 cv26_20558b6a5c8dbac9 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22004016.mp3      5.621625                     另一联赛是阿根廷全国足球甲级联赛。 pending
              4 cv26_40123f9681ffefa4 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_18994260

In [6]:
#展示10条音频
for i in range(1,11):
    show_candidate(i)

序号: 1
UID: cv26_3f66c228bc2a4ae8
音频时长: 5.952
当前状态: pending
原始转写: 这两家公司都在网上出售树莓派。


序号: 2
UID: cv26_1753be3b6083f28f
音频时长: 2.784
当前状态: pending
原始转写: 武藏国出身。


序号: 3
UID: cv26_20558b6a5c8dbac9
音频时长: 5.616
当前状态: pending
原始转写: 另一联赛是阿根廷全国足球甲级联赛。


序号: 4
UID: cv26_40123f9681ffefa4
音频时长: 6.384
当前状态: pending
原始转写: 措布伦是奥地利蒂罗尔州罗伊特县的一个市镇。


序号: 5
UID: cv26_8e8d4f65babf0b7e
音频时长: 8.976
当前状态: pending
原始转写: 维济马农村居民点是俄罗斯联邦沃洛格达州别洛泽尔斯克区所属的一个农村居民点。


序号: 6
UID: cv26_95b35d7219a5eeb9
音频时长: 5.016
当前状态: pending
原始转写: 也是日本电视控股事业的子公司之一。


序号: 7
UID: cv26_dc3e083eab508fd5
音频时长: 4.392
当前状态: pending
原始转写: 编组共两节车厢。


序号: 8
UID: cv26_f05564f22bc1d77c
音频时长: 6.576
当前状态: pending
原始转写: 德国空军将西班牙内战学习的经验第一次运用在大规模战争上。


序号: 9
UID: cv26_2bd1b28412804c60
音频时长: 7.956
当前状态: pending
原始转写: 在接下来的几个月里，“信天翁”在旧金山境外运营。


序号: 10
UID: cv26_1441a8cec768fc20
音频时长: 8.496
当前状态: pending
原始转写: 阿韦利诺洛佩斯是巴西皮奥伊州的一个市镇。


In [10]:
#标注更新
from pathlib import Path
import sys

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

from src.normalize import normalize_score

update_candidate(
    1,
    raw_text="这两家公司都在网上出售树莓派。",
    status="verified",
    score_text=normalize_score("这两家公司都在网上出售树莓派。"),
    review_minutes="0.5"
)

update_candidate(
    2,
    raw_text="武藏国出身。",
    status="verified",
    score_text=normalize_score("武藏国出身。"),
    review_minutes="0.3"
)

update_candidate(
    3,
    raw_text="另一联赛是阿根廷全国足球甲级联赛。",
    status="verified",
    score_text=normalize_score("另一联赛是阿根廷全国足球甲级联赛。"),
    review_minutes="0.4"
)

update_candidate(
    4,
    raw_text="措布伦是奥地利蒂罗尔州罗伊特县的一个市镇。",
    status="verified",
    score_text=normalize_score("措布伦是奥地利蒂罗尔州罗伊特县的一个市镇。"),
    review_minutes="0.5"
)

update_candidate(
    5,
    raw_text="维济马农村居民点是俄罗斯联邦沃洛格达州别洛泽尔斯克区所属的一个农村居民点。",
    status="verified",
    score_text=normalize_score("维济马农村居民点是俄罗斯联邦沃洛格达州别洛泽尔斯克区所属的一个农村居民点。"),
    review_minutes="0.8"
)

update_candidate(
    6,
    raw_text="也是日本电视控股事业的[uncertain]公司之一。",
    status="needs_review",
    score_text=normalize_score(""),
    review_minutes="1.0",
    uncertain_span="事业的与公司之间存在一个音节，但无法辨认是否为子",
    notes="录音质量问题，反复回听仍无法确认具体字词，暂不通过验收。"
)

update_candidate(
    7,
    raw_text="编组共两节车厢。",
    status="verified",
    score_text=normalize_score("编组共两节车厢。"),
    review_minutes="0.4"
)

update_candidate(
    8,
    raw_text="德国空军将西班牙内战学习的经验第一次运用在大规模战争上。",
    status="verified",
    score_text=normalize_score("德国空军将西班牙内战学习的经验第一次运用在大规模战争上。"),
    review_minutes="0.5"
)

update_candidate(
    9,
    raw_text="在接下来的几个月里，“信天翁”在旧金山境外运营。",
    status="verified",
    score_text=normalize_score("在接下来的几个月里，“信天翁”在旧金山境外运营。"),
    review_minutes="0.5"
)

update_candidate(
    10,
    raw_text="阿韦利诺洛佩斯是巴西皮奥伊州的一个市镇。",
    status="verified",
    score_text=normalize_score("阿韦利诺洛佩斯是巴西皮奥伊州的一个市镇。"),
    review_minutes="0.5"
)

第1条已更新为：verified
第2条已更新为：verified
第3条已更新为：verified
第4条已更新为：verified
第5条已更新为：verified
第6条已更新为：needs_review
第7条已更新为：verified
第8条已更新为：verified
第9条已更新为：verified
第10条已更新为：verified


In [12]:
#由于第6条无法辨认，所以加一条11
show_candidate(11)
update_candidate(
    11,
    raw_text="母孙氏。",
    status="verified",
    score_text=normalize_score("母孙氏。"),
    review_minutes="0.4"
)

序号: 11
UID: cv26_dc1278883fe4eb06
音频时长: 2.784
当前状态: pending
原始转写: 母孙氏。


第11条已更新为：verified


In [13]:
#检查本批次状态
import pandas as pd

path = (
    r"D:\Projects\speech-ai-learning"
    r"\metadata\dev_candidates_W7D5.csv"
)

df = pd.read_csv(path)

print("状态统计：")
print(df["status"].value_counts())

print("\n字段检查：")
for col in [
    "raw_text",
    "score_text",
    "review_minutes"
]:
    print(
        f"{col} 为空：",
        df.loc[
            df["status"] == "verified", col
        ].isna().sum()
    )

print(
    "\n累计 Dev verified:",
    (df["status"] == "verified").sum()
)

assert (
    df["status"] == "verified"
).sum() == 10

verified_df = df[df["status"] == "verified"]

assert verified_df[
    ["raw_text", "score_text", "review_minutes"]
].notna().all().all()

assert (
    verified_df["raw_text"].astype(str).str.strip() != ""
).all()

assert (
    verified_df["score_text"].astype(str).str.strip() != ""
).all()

print("\nW7D5 Dev 标注验收通过")

状态统计：
status
pending         398
verified         10
needs_review      1
Name: count, dtype: int64

字段检查：
raw_text 为空： 0
score_text 为空： 0
review_minutes 为空： 0

累计 Dev verified: 10

W7D5 Dev 标注验收通过


### 3.更新12-21条候选队列

In [9]:
#检验队列
import pandas as pd

path = (
    r"D:\Projects\speech-ai-learning"
    r"\metadata\dev_candidates_W7D5.csv"
)

df = pd.read_csv(path)

batch = df[
    df["selection_rank"].between(12,21)
].copy()

print(
    batch[
        [
            "selection_rank",
            "candidate_uid",
            "audio_path",
            "duration_sec",
            "source_text",
            "status"
        ]
    ].to_string(index=False)
)

assert len(batch) == 10

 selection_rank         candidate_uid                                                                                                         audio_path  duration_sec                           source_text  status
             12 cv26_04fa4b16b8e447e0 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22148152.mp3      3.293625                            拉威茨出生在洛杉矶。 pending
             13 cv26_cd2999a943f09d9b D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_23607346.mp3      3.701625                                辽朝契丹人。 pending
             14 cv26_80f290bef356a65b D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_26066531.mp3      8.575500 从地质学上看，这座山由 Kinder Scout Grit 这种砂砾构成。 pending
             15 cv26_4866c657ad16cbfd D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22057330

In [10]:
#展示10条音频
for i in range(12,22):
    show_candidate(i)

序号: 12
UID: cv26_04fa4b16b8e447e0
音频时长: 3.288
当前状态: pending
原始转写: 拉威茨出生在洛杉矶。


序号: 13
UID: cv26_cd2999a943f09d9b
音频时长: 3.696
当前状态: pending
原始转写: 辽朝契丹人。


序号: 14
UID: cv26_80f290bef356a65b
音频时长: 8.568
当前状态: pending
原始转写: 从地质学上看，这座山由 Kinder Scout Grit 这种砂砾构成。


序号: 15
UID: cv26_4866c657ad16cbfd
音频时长: 4.464
当前状态: pending
原始转写: 这引起了国际社会的反响。


序号: 16
UID: cv26_fd55b4d908f51022
音频时长: 2.688
当前状态: pending
原始转写: 桑杜斯。


序号: 17
UID: cv26_e20a7065b0f47587
音频时长: 4.248
当前状态: pending
原始转写: 小裁缝一个手掌把其中的七只打死了。


序号: 18
UID: cv26_2e8b61252f53e7c2
音频时长: 3.792
当前状态: pending
原始转写: 鲇属是鲇科的一个属。


序号: 19
UID: cv26_c8a336cecf0eeaf1
音频时长: 8.832
当前状态: pending
原始转写: 下湖是位于美国加利福尼亚州莱克县的一个人口普查指定地区。


序号: 20
UID: cv26_496f7a48e0d64d24
音频时长: 7.44
当前状态: pending
原始转写: 嘉庆七年壬戌科第三甲第九十六名同进士出身。


序号: 21
UID: cv26_b4558f9bf3d1e9aa
音频时长: 2.736
当前状态: pending
原始转写: 发行商为。


In [11]:
#标注更新
from pathlib import Path
import sys

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

from src.normalize import normalize_score

update_candidate(
    12,
    raw_text="拉威茨出生在洛杉矶。",
    status="verified",
    score_text=normalize_score("拉威茨出生在洛杉矶。"),
    review_minutes="0.3"
)

update_candidate(
    13,
    raw_text="辽朝契丹人。",
    status="verified",
    score_text=normalize_score("辽朝契丹人。"),
    review_minutes="0.3"
)

update_candidate(
    14,
    raw_text="从地质学上看，这座山由 Kinder Scout Grit 这种砂砾构成。",
    status="verified",
    score_text=normalize_score("从地质学上看，这座山由 Kinder Scout Grit 这种砂砾构成。"),
    review_minutes="0.5"
)

update_candidate(
    15,
    raw_text="这引起了国际社会的反响。",
    status="verified",
    score_text=normalize_score("这引起了国际社会的反响。"),
    review_minutes="0.5"
)

update_candidate(
    16,
    raw_text="桑杜斯。",
    status="verified",
    score_text=normalize_score("桑杜斯。"),
    review_minutes="0.2"
)

update_candidate(
    17,
    raw_text="小裁缝一个手掌把其中的七只打死了。",
    status="verified",
    score_text=normalize_score("小裁缝一个手掌把其中的七只打死了。"),
    review_minutes="0.4"
)

update_candidate(
    18,
    raw_text="鲇属是鲇科的一个属。",
    status="verified",
    score_text=normalize_score("鲇属是鲇科的一个属。"),
    review_minutes="0.4"
)

update_candidate(
    19,
    raw_text="下湖是位于美国加利福尼亚州莱克县的一个人口普查指定地区。",
    status="verified",
    score_text=normalize_score("下湖是位于美国加利福尼亚州莱克县的一个人口普查指定地区。"),
    review_minutes="0.5"
)

update_candidate(
    20,
    raw_text="嘉庆七年壬戌科第三甲第九十六名同进士出身。",
    status="verified",
    score_text=normalize_score("嘉庆七年壬戌科第三甲第九十六名同进士出身。"),
    review_minutes="0.5"
)

update_candidate(
    21,
    raw_text="发行商为。",
    status="verified",
    score_text=normalize_score("发行商为。"),
    review_minutes="0.3"
)

第12条已更新为：verified
第13条已更新为：verified
第14条已更新为：verified
第15条已更新为：verified
第16条已更新为：verified
第17条已更新为：verified
第18条已更新为：verified
第19条已更新为：verified
第20条已更新为：verified
第21条已更新为：verified


In [12]:
#完成后统计输出
import pandas as pd

path = (
    r"D:\Projects\speech-ai-learning"
    r"\metadata\dev_candidates_W7D5.csv"
)

df = pd.read_csv(path)

batch = df[
    df["selection_rank"].between(12, 21)
]

print("状态统计：")
print(batch["status"].value_counts())

print("\n字段检查：")
for col in ["raw_text", "score_text", "review_minutes"]:
    print(f"{col} 为空：", batch[col].isna().sum())

print(
    "\n累计 verified:",
    (df["status"] == "verified").sum()
)

状态统计：
status
verified    10
Name: count, dtype: int64

字段检查：
raw_text 为空： 0
score_text 为空： 0
review_minutes 为空： 0

累计 verified: 20


### 4.更新22-31条候选队列

In [3]:
#检验队列
import pandas as pd

path = (
    r"D:\Projects\speech-ai-learning"
    r"\metadata\dev_candidates_W7D5.csv"
)

df = pd.read_csv(path)

batch = df[
    df["selection_rank"].between(22,31)
].copy()

print(
    batch[
        [
            "selection_rank",
            "candidate_uid",
            "audio_path",
            "duration_sec",
            "source_text",
            "status"
        ]
    ].to_string(index=False)
)

assert len(batch) == 10

 selection_rank         candidate_uid                                                                                                         audio_path  duration_sec                     source_text  status
             22 cv26_e89642a3e548d5d8 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22253359.mp3      6.941625              该出口亦设有连通南口剪票口层的电梯。 pending
             23 cv26_78ec898f36aaa003 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_18641896.mp3      5.957625           在不同年份，欧洲锦标赛使用不同的比赛模式。 pending
             24 cv26_1ac46542f8b14712 D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_18978225.mp3      5.453625                澳门五月诗社是一个澳门文学团体。 pending
             25 cv26_0db64898e657206e D:\Projects\speech-ai-learning\data\raw\common_voice_beijing26\zh-CN-beijing\clips\common_voice_zh-CN_22132148.mp3      6.629625      

In [4]:
#展示10条音频
for i in range(22,32):
    show_candidate(i)

序号: 22
UID: cv26_e89642a3e548d5d8
音频时长: 6.936
当前状态: pending
原始转写: 该出口亦设有连通南口剪票口层的电梯。


序号: 23
UID: cv26_78ec898f36aaa003
音频时长: 5.952
当前状态: pending
原始转写: 在不同年份，欧洲锦标赛使用不同的比赛模式。


序号: 24
UID: cv26_1ac46542f8b14712
音频时长: 5.448
当前状态: pending
原始转写: 澳门五月诗社是一个澳门文学团体。


序号: 25
UID: cv26_0db64898e657206e
音频时长: 6.624
当前状态: pending
原始转写: 特殊奥林匹克运动是智力残疾人参与的体育运动。


序号: 26
UID: cv26_70240f012fa3cd9c
音频时长: 4.56
当前状态: pending
原始转写: 根据国际特赦组织。


序号: 27
UID: cv26_5db101e331fbf5b1
音频时长: 4.104
当前状态: pending
原始转写: 教区位于上奥地利州。


序号: 28
UID: cv26_30954e3ebeb93340
音频时长: 9.84
当前状态: pending
原始转写: 逆康普顿散射也可以看作是相对论性粒子自身参考系下的汤姆孙散射。


序号: 29
UID: cv26_0fabf67edba595ad
音频时长: 3.504
当前状态: pending
原始转写: 亦新增纹理阵列功能。


序号: 30
UID: cv26_8fc574ded5662a4c
音频时长: 9.768
当前状态: pending
原始转写: 近红外线照相机和多目标分光仪上使用于红外线天文学的科学仪器。


序号: 31
UID: cv26_13e49e69242ee6e7
音频时长: 5.58
当前状态: pending
原始转写: 总部设在卡萨布兰卡穆罕默德五世国际机场。


In [5]:
#标注更新
from pathlib import Path
import sys

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

from src.normalize import normalize_score

update_candidate(
    22,
    raw_text="该出口亦设有连通南口检票口层的电梯。",
    status="verified",
    score_text=normalize_score("该出口亦设有连通南口检票口层的电梯。"),
    review_minutes="0.3"
)

update_candidate(
    23,
    raw_text="在不同年份，欧洲锦标赛使用不同的比赛模式。",
    status="verified",
    score_text=normalize_score("在不同年份，欧洲锦标赛使用不同的比赛模式。"),
    review_minutes="0.4"
)

update_candidate(
    24,
    raw_text="澳门五月诗社是一个澳门文学团体。",
    status="verified",
    score_text=normalize_score("澳门五月诗社是一个澳门文学团体。"),
    review_minutes="0.3"
)

update_candidate(
    25,
    raw_text="特殊奥林匹克运动是智力残疾人参与的体育运动。",
    status="verified",
    score_text=normalize_score("特殊奥林匹克运动是智力残疾人参与的体育运动。"),
    review_minutes="0.4"
)

update_candidate(
    26,
    raw_text="根据国际特赦组织。",
    status="verified",
    score_text=normalize_score("根据国际特赦组织。"),
    review_minutes="0.2"
)

update_candidate(
    27,
    raw_text="教区位于上奥地利州。",
    status="verified",
    score_text=normalize_score("教区位于上奥地利州。"),
    review_minutes="0.2"
)

update_candidate(
    28,
    raw_text="逆康普顿散射也可以看作是相对论性粒子自身参考系下的汤姆孙散射。",
    status="verified",
    score_text=normalize_score("逆康普顿散射也可以看作是相对论性粒子自身参考系下的汤姆孙散射。"),
    review_minutes="0.5"
)

update_candidate(
    29,
    raw_text="亦新增纹理阵列功能。",
    status="verified",
    score_text=normalize_score("亦新增纹理阵列功能。"),
    review_minutes="0.3"
)

update_candidate(
    30,
    raw_text="近红外线照相机和多目标分光仪上使用于红外线天文学的科学仪器。",
    status="verified",
    score_text=normalize_score("近红外线照相机和多目标分光仪上使用于红外线天文学的科学仪器。"),
    review_minutes="0.5"
)

update_candidate(
    31,
    raw_text="总部设在卡萨布兰卡穆罕默德五世国际机场。",
    status="verified",
    score_text=normalize_score("总部设在卡萨布兰卡穆罕默德五世国际机场。"),
    review_minutes="0.4"
)

第22条已更新为：verified
第23条已更新为：verified
第24条已更新为：verified
第25条已更新为：verified
第26条已更新为：verified
第27条已更新为：verified
第28条已更新为：verified
第29条已更新为：verified
第30条已更新为：verified
第31条已更新为：verified


In [6]:
#完成后统计输出
import pandas as pd

path = (
    r"D:\Projects\speech-ai-learning"
    r"\metadata\dev_candidates_W7D5.csv"
)

df = pd.read_csv(path)

batch = df[
    df["selection_rank"].between(22, 31)
]

print("状态统计：")
print(batch["status"].value_counts())

print("\n字段检查：")
for col in ["raw_text", "score_text", "review_minutes"]:
    print(f"{col} 为空：", batch[col].isna().sum())

print(
    "\n累计 verified:",
    (df["status"] == "verified").sum()
)

状态统计：
status
verified    10
Name: count, dtype: int64

字段检查：
raw_text 为空： 0
score_text 为空： 0
review_minutes 为空： 0

累计 verified: 30


### 5.补齐Dev元数据

In [9]:
# 只保留已经校对通过的 Dev
df = df[df["status"] == "verified"].copy()

print("Dev verified 数量:", len(df))
print("UID 唯一数量:", df["candidate_uid"].nunique())

Dev verified 数量: 30
UID 唯一数量: 30


In [ ]:
print("===== Dev audio_path =====")
print(df["audio_path"].head(5).tolist())

print("\n===== Manifest audio_path =====")
print(manifest["audio_path"].head(5).tolist())

print("\n===== Exact Match =====")

print(
    "匹配数量:",
    df["audio_path"].astype(str).str.strip().isin(
        manifest["audio_path"].astype(str).str.strip()
    ).sum()
)

===== Dev audio_path =====
['D:\\Projects\\speech-ai-learning\\data\\raw\\common_voice_beijing26\\zh-CN-beijing\\clips\\common_voice_zh-CN_22057623.mp3', 'D:\\Projects\\speech-ai-learning\\data\\raw\\common_voice_beijing26\\zh-CN-beijing\\clips\\common_voice_zh-CN_22004281.mp3', 'D:\\Projects\\speech-ai-learning\\data\\raw\\common_voice_beijing26\\zh-CN-beijing\\clips\\common_voice_zh-CN_22004016.mp3', 'D:\\Projects\\speech-ai-learning\\data\\raw\\common_voice_beijing26\\zh-CN-beijing\\clips\\common_voice_zh-CN_18994260.mp3', 'D:\\Projects\\speech-ai-learning\\data\\raw\\common_voice_beijing26\\zh-CN-beijing\\clips\\common_voice_zh-CN_18978223.mp3']

===== Manifest audio_path =====
['common_voice_zh-CN_27897382.mp3', 'common_voice_zh-CN_28304063.mp3', 'common_voice_zh-CN_28375337.mp3', 'common_voice_zh-CN_28375320.mp3', 'common_voice_zh-CN_28374955.mp3']

===== Exact Match =====
匹配数量: 0


In [12]:
# Dev：提取文件名
df["audio_name"] = (
    df["audio_path"]
    .astype("string")
    .str.replace("\\", "/", regex=False)
    .str.rsplit("/", n=1)
    .str[-1]
)

# Manifest：提取文件名
manifest["audio_name"] = (
    manifest["audio_path"]
    .astype("string")
    .str.replace("\\", "/", regex=False)
    .str.rsplit("/", n=1)
    .str[-1]
)

# 检查 Manifest 文件名是否唯一
print(
    "Manifest 文件名重复数量:",
    manifest["audio_name"].duplicated().sum()
)

# 检查匹配数量
print(
    "Dev 与 Manifest 匹配数量:",
    df["audio_name"].isin(
        manifest["audio_name"]
    ).sum()
)

# 查看未匹配文件名
print("\n未匹配文件名:")
print(
    df.loc[
        ~df["audio_name"].isin(manifest["audio_name"]),
        "audio_name"
    ].tolist()
)

Manifest 文件名重复数量: 0
Dev 与 Manifest 匹配数量: 30

未匹配文件名:
[]


In [13]:
#正式补齐元数据
# 检查 Dev 文件名是否唯一
print(
    "Dev 文件名重复数量:",
    df["audio_name"].duplicated().sum()
)

# 关联 Manifest 元数据
df = df.merge(
    manifest[
        [
            "audio_name",
            "source_id",
            "start_time",
            "end_time"
        ]
    ],
    on="audio_name",
    how="left",
    validate="one_to_one",
    indicator=True
)

# 检查关联结果
print("\n===== Dev Metadata Join =====")
print(df["_merge"].value_counts())

print("\n===== Metadata Missing =====")
print(
    df[
        ["source_id", "start_time", "end_time"]
    ].isna().sum()
)

print("\n===== Dev Preview =====")
print(
    df[
        [
            "candidate_uid",
            "audio_name",
            "source_id",
            "start_time",
            "end_time"
        ]
    ].head()
)

Dev 文件名重复数量: 0

===== Dev Metadata Join =====
_merge
both          30
left_only      0
right_only     0
Name: count, dtype: int64

===== Metadata Missing =====
source_id     0
start_time    0
end_time      0
dtype: int64

===== Dev Preview =====
           candidate_uid                       audio_name  \
0  cv26_3f66c228bc2a4ae8  common_voice_zh-CN_22057623.mp3   
1  cv26_1753be3b6083f28f  common_voice_zh-CN_22004281.mp3   
2  cv26_20558b6a5c8dbac9  common_voice_zh-CN_22004016.mp3   
3  cv26_40123f9681ffefa4  common_voice_zh-CN_18994260.mp3   
4  cv26_8e8d4f65babf0b7e  common_voice_zh-CN_18978223.mp3   

                      source_id  start_time  end_time  
0  common_voice_beijing26_zh-CN         0.0  5.957625  
1  common_voice_beijing26_zh-CN         0.0  2.789625  
2  common_voice_beijing26_zh-CN         0.0  5.621625  
3  common_voice_beijing26_zh-CN         0.0  6.389625  
4  common_voice_beijing26_zh-CN         0.0  8.981625  


In [14]:
import pandas as pd

# 读取 Test 候选文件
candidates = pd.read_csv(
    r"D:\Projects\speech-ai-learning\metadata\test_candidates.csv",
    keep_default_na=False
)

# 仅保留已经人工校对通过的 Test
candidates = candidates[
    candidates["status"] == "verified"
].copy()

# 统一 Test 的音频文件名
candidates["audio_name"] = (
    candidates["audio_path"]
    .astype("string")
    .str.replace("\\", "/", regex=False)
    .str.rsplit("/", n=1)
    .str[-1]
)

print("===== Dataset Summary =====")
print("Test verified:", len(candidates))
print("Dev verified:", len(df))

print("\nTest split:")
print(candidates["split"].value_counts())

print("\nDev split:")
print(df["split"].value_counts())

===== Dataset Summary =====
Test verified: 90
Dev verified: 30

Test split:
split
test    90
Name: count, dtype: int64

Dev split:
split
dev    30
Name: count, dtype: int64


In [15]:
# 检查四个字段
for column in [
    "candidate_uid",
    "audio_name",
    "parent_uid",
    "speaker_id"
]:
    test_values = set(
        candidates[column].dropna().astype(str).str.strip()
    )

    dev_values = set(
        df[column].dropna().astype(str).str.strip()
    )

    test_values.discard("")
    dev_values.discard("")

    overlap = test_values & dev_values

    print(f"\n===== {column} =====")
    print("Test 唯一数量:", len(test_values))
    print("Dev 唯一数量:", len(dev_values))
    print("交叠数量:", len(overlap))

    if overlap:
        print("交叠示例:", list(overlap)[:10])


===== candidate_uid =====
Test 唯一数量: 90
Dev 唯一数量: 30
交叠数量: 0

===== audio_name =====
Test 唯一数量: 90
Dev 唯一数量: 30
交叠数量: 0

===== parent_uid =====
Test 唯一数量: 90
Dev 唯一数量: 30
交叠数量: 0

===== speaker_id =====
Test 唯一数量: 42
Dev 唯一数量: 17
交叠数量: 0
